## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [3]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("hf_token_rp")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [4]:
import os
os.environ.setdefault("USE_TF", "0")
import random
import json
import time
import math
import zipfile
import shutil

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# You can use any of the 4 train files or all the files for pre-training the legal model
TRAIN_FILE0 = "Phase_1_Pretraining/Train/train_supreme_court.json"
TRAIN_FILE1 = "Phase_1_Pretraining/Train/train_high_courts.json"
TRAIN_FILE2 = "Phase_1_Pretraining/Train/train_acts.json"
TRAIN_FILE3 = "Phase_1_Pretraining/Train/train_tribunals.json"

TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TRAIN_TOKENS = 200_000_000
VAL_TOKENS = 5_000_000
TRAIN_BLOCK_SIZE = 2048
EPOCHS = 3
SEED = 42
random.seed(SEED)
MODEL_CONTEXT_LENGTH = 2048
N_EMBD = 640
N_LAYER = 8
N_HEAD = 8

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95

# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-decoder-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")

# ------------------------------------------------------------
# Run-mode switches
# ------------------------------------------------------------
# False: load the saved BPE tokenizer used for every submission (recommended).
# True : retrain the BPE tokenizer from the corpus (slow, pure Python).
TRAIN_NEW_TOKENIZER = False
TOKENIZER_INPUT_PATH = "/kaggle/input/datasets/sg123321/tokenizer/legal_bpe.json"

# Optional pre-tokenized corpus (train/val token arrays). If it is not attached,
# the mixed corpus is built and tokenized in this notebook and cached in OUTPUT_DIR.
TOKENIZED_CACHE_INPUT = "/kaggle/input/datasets/sg123321/tokenizer/tokenized_data.npz"

# Full-corpus token statistics (used for the report). Very slow; off by default.
COUNT_CORPUS_TOKENS = False

# Resume an interrupted run from a Trainer checkpoint directory (None = start from random init).
RESUME_FROM_CHECKPOINT = None


Configuration loaded.


## 4. Download Training Data From Hugging Face

In [5]:
from huggingface_hub import hf_hub_download

# Download all four training files
TRAIN_FILES = {
    "supreme_court": hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=TRAIN_FILE0,
        repo_type="dataset"
    ),
    "high_courts": hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=TRAIN_FILE1,
        repo_type="dataset"
    ),
    "acts": hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=TRAIN_FILE2,
        repo_type="dataset"
    ),
    "tribunals": hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=TRAIN_FILE3,
        repo_type="dataset"
    )
}

for source, path in TRAIN_FILES.items():
    print(f"{source}: {path}")

Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

supreme_court: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json
high_courts: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json
acts: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json
tribunals: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json


## 5. Load Tokenizer

In [5]:

# tokenizer.pad_token = tokenizer.eos_token
# print("Vocabulary size:", len(tokenizer))

#BPE tokenizer

def train_tokenizer(train_text, vocab_size):
    tokenizer = ByteLevelBPE(vocab_size=vocab_size)
    tokenizer.train([train_text])
    return tokenizer

# ============================================================
# QUESTION 5 — OPTIMIZED BYTE-LEVEL BPE
# ============================================================

import json
from collections import Counter, defaultdict
from tqdm.auto import tqdm


BPE_SPECIALS = [
    "<pad>",
    "<unk>",
    "<s>",
    "</s>"
]

# End-of-word marker.
# UTF-8 bytes are integers 0..255, so 256 cannot collide.
EOW = 256


class ByteLevelBPE:
    """
    Deterministic byte-level BPE tokenizer.

    - Initial units are UTF-8 bytes.
    - EOW marks the end of each word.
    - Case is preserved.
    - BPE never crosses word boundaries.
    - Ties are broken lexicographically.
    - Four special tokens are fixed at the beginning.
    """

    def __init__(self, vocab_size=5000):

        self.vocab_size = vocab_size
        self.merge_ranks = {}
        self.specials = list(BPE_SPECIALS)

        # List of learned merges.
        #
        # Each element is:
        #     (left_symbol, right_symbol)
        #
        self.merges = []

        # Vocabulary.
        self.itos = []
        self.stoi = {}

        # Internal merge rules.
        #
        # (left, right) -> merged_symbol
        #
        self.merge_rules = {}

        # Final representations of words seen during training.
        self.word_cache = {}
        self.token_to_symbol = {}

    # --------------------------------------------------------
    # Convert a byte/symbol tuple into a deterministic string
    # --------------------------------------------------------
    @staticmethod
    def string_to_symbol(token):
        # Standalone end-of-word marker
        if token == "</w>":
            return (EOW,)
    
        # Merged symbol ending with EOW
        if token.endswith("</w>>"):
            hex_part = token[3:-5]
            byte_part = tuple(
                int(hex_part[i:i+2], 16)
                for i in range(0, len(hex_part), 2)
            )
            return byte_part + (EOW,)
    
        # Ordinary byte or merged byte symbol
        hex_part = token[3:-1]
        return tuple(
            int(hex_part[i:i+2], 16)
            for i in range(0, len(hex_part), 2)
        )
    @staticmethod
    def symbol_to_string(symbol):

        # EOW is always the last symbol when present.
        if symbol and symbol[-1] == EOW:

            byte_part = symbol[:-1]

            if len(byte_part) == 0:
                return "</w>"

            hex_part = "".join(
                f"{b:02x}"
                for b in byte_part
            )

            return f"<0x{hex_part}</w>>"

        hex_part = "".join(
            f"{b:02x}"
            for b in symbol
        )

        return f"<0x{hex_part}>"
    def _build_merge_ranks(self):
        self.merge_ranks = {
            pair: rank
            for rank, pair in enumerate(self.merge_rules)
        }
    # --------------------------------------------------------
    # Deterministic ordering key for symbols
    # --------------------------------------------------------

    @staticmethod
    def symbol_key(symbol):
        return tuple(symbol)

    # --------------------------------------------------------
    # Convert a word into initial UTF-8 byte symbols
    # --------------------------------------------------------

    @staticmethod
    def word_to_symbols(word):

        byte_values = list(
            word.encode("utf-8")
        )

        symbols = [
            (b,)
            for b in byte_values
        ]

        # Explicit word boundary.
        symbols.append((EOW,))

        return symbols

    # --------------------------------------------------------
    # Count all pairs once initially
    # --------------------------------------------------------

    def _initial_pair_counts(
        self,
        word_symbols,
        word_counts
    ):

        pair_counts = Counter()

        # pair_words tells us which words contain a pair.
        pair_words = defaultdict(set)

        for word, symbols in word_symbols.items():

            freq = word_counts[word]

            for i in range(len(symbols) - 1):

                pair = (
                    symbols[i],
                    symbols[i + 1]
                )

                pair_counts[pair] += freq
                pair_words[pair].add(word)

        return pair_counts, pair_words

    # --------------------------------------------------------
    # Merge one pair in one word
    # --------------------------------------------------------

    @staticmethod
    def _merge_word(symbols, pair):

        left, right = pair

        new_symbol = left + right

        result = []

        i = 0

        while i < len(symbols):

            if (
                i + 1 < len(symbols)
                and symbols[i] == left
                and symbols[i + 1] == right
            ):

                result.append(new_symbol)

                i += 2

            else:

                result.append(symbols[i])

                i += 1

        return result

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    def train(self, texts):

        print("=" * 70)
        print("Training BYTE-LEVEL BPE")
        print("=" * 70)

        # ----------------------------------------------------
        # Count unique words.
        # ----------------------------------------------------

        word_counts = Counter()

        for text in texts:
            if not isinstance(text, str):
                continue

            for word in text.split():

                word_counts[word] += 1

        print(
            f"Unique words: {len(word_counts):,}"
        )

        # ----------------------------------------------------
        # Initial representation of each word.
        # ----------------------------------------------------

        word_symbols = {
            word: self.word_to_symbols(word)
            for word in word_counts
        }

        # ----------------------------------------------------
        # Initial vocabulary symbols.
        # ----------------------------------------------------

        initial_symbols = [
            (b,)
            for b in range(256)
        ]

        initial_symbols.append((EOW,))

        # Four specials + initial byte symbols.
        current_vocab_size = (
            len(BPE_SPECIALS)
            + len(initial_symbols)
        )

        print(
            f"Initial vocabulary size: "
            f"{current_vocab_size:,}"
        )

        if current_vocab_size > self.vocab_size:

            raise ValueError(
                "Initial byte vocabulary already exceeds "
                "the requested vocabulary size."
            )

        # ----------------------------------------------------
        # Count pairs once.
        # ----------------------------------------------------

        pair_counts, pair_words = \
            self._initial_pair_counts(
                word_symbols,
                word_counts
            )

        # ----------------------------------------------------
        # Main BPE loop.
        # ----------------------------------------------------

        merge_no = 0

        while current_vocab_size < self.vocab_size:

            if not pair_counts:
                break

            # Ignore pairs occurring only once.
            valid_pairs = [
                pair
                for pair, count in pair_counts.items()
                if count > 1
            ]

            if not valid_pairs:
                print(
                    "No pair occurs more than once. "
                    "Stopping early."
                )
                break

            # Highest frequency first.
            # Lexicographically smallest pair on ties.
            best_pair = min(
                valid_pairs,
                key=lambda pair: (
                    -pair_counts[pair],
                    self.symbol_key(pair[0]),
                    self.symbol_key(pair[1])
                )
            )

            best_count = pair_counts[best_pair]

            left, right = best_pair
            new_symbol = left + right

            # ------------------------------------------------
            # Record merge.
            # ------------------------------------------------

            self.merge_rules[best_pair] = new_symbol

            self.merges.append(
                (
                    self.symbol_to_string(left),
                    self.symbol_to_string(right)
                )
            )

            # ------------------------------------------------
            # Words affected by this merge.
            # ------------------------------------------------

            affected_words = list(
                pair_words.get(best_pair, set())
            )

            # Remove the selected pair's old contribution.
            del pair_counts[best_pair]

            pair_words.pop(
                best_pair,
                None
            )

            # ------------------------------------------------
            # Update each affected word.
            # ------------------------------------------------

            for word in affected_words:

                old_symbols = word_symbols[word]

                old_pairs = []

                for i in range(
                    len(old_symbols) - 1
                ):

                    old_pairs.append(
                        (
                            old_symbols[i],
                            old_symbols[i + 1]
                        )
                    )

                # Remove old pair contributions.
                freq = word_counts[word]

                for pair in old_pairs:

                    pair_counts[pair] -= freq

                    if pair_counts[pair] <= 0:
                        pair_counts.pop(
                            pair,
                            None
                        )

                    if pair in pair_words:
                        pair_words[pair].discard(word)

                        if not pair_words[pair]:
                            pair_words.pop(
                                pair,
                                None
                            )

                # Perform merge.
                new_symbols = self._merge_word(
                    old_symbols,
                    best_pair
                )

                word_symbols[word] = new_symbols

                # Add new pair contributions.
                new_pairs = []

                for i in range(
                    len(new_symbols) - 1
                ):

                    new_pairs.append(
                        (
                            new_symbols[i],
                            new_symbols[i + 1]
                        )
                    )

                for pair in new_pairs:

                    pair_counts[pair] += freq
                    pair_words[pair].add(word)

            current_vocab_size += 1
            merge_no += 1

            if merge_no % 250 == 0:

                print(
                    f"Merges: {merge_no:4d} | "
                    f"Vocab: {current_vocab_size:5d} | "
                    f"Best pair frequency: {best_count:,}"
                )

        # ----------------------------------------------------
        # Construct final vocabulary.
        #
        # First:
        #   <pad>, <unk>, <s>, </s>
        #
        # Then initial symbols.
        #
        # Then symbols introduced by merges in merge order.
        # ----------------------------------------------------

        merged_symbols = list(
            self.merge_rules.values()
        )

        self.itos = (
            list(BPE_SPECIALS)
            +
            [
                self.symbol_to_string(s)
                for s in initial_symbols
            ]
            +
            [
                self.symbol_to_string(s)
                for s in merged_symbols
            ]
        )

        self.token_to_symbol = {}

        for symbol in initial_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        for symbol in merged_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        # Remove duplicates while preserving order.
        final_itos = []
        seen = set()

        for token in self.itos:

            if token not in seen:

                seen.add(token)
                final_itos.append(token)

        self.itos = final_itos[
            :self.vocab_size
        ]

        self.stoi = {
            token: i
            for i, token in enumerate(self.itos)
        }

        # Save representations of training words.
        self.word_cache = word_symbols

        print(
            "\nTraining complete."
        )

        print(
            f"Final vocabulary size: "
            f"{len(self.itos):,}"
        )

        print(
            f"Number of merges: "
            f"{len(self.merges):,}"
        )
        self._build_merge_ranks()
        return self

    # --------------------------------------------------------
    # Tokenize a word
    # --------------------------------------------------------

    def tokenize_word(self, word):
        # Use cached representation for training words.
        if word in self.word_cache:
            symbols = self.word_cache[word]
        else:
            symbols = self.word_to_symbols(word)
    
            while len(symbols) > 1:
                # Find adjacent pairs that have learned merges.
                candidates = [
                    (self.merge_ranks[(symbols[i], symbols[i + 1])], i)
                    for i in range(len(symbols) - 1)
                    if (symbols[i], symbols[i + 1]) in self.merge_ranks
                ]
    
                if not candidates:
                    break
    
                # Apply the earliest learned merge.
                _, best_idx = min(candidates)
    
                pair = (symbols[best_idx], symbols[best_idx + 1])
                new_symbol = self.merge_rules[pair]
    
                # Merge all non-overlapping occurrences.
                symbols = self._merge_word(symbols, pair)
    
        return [
            self.symbol_to_string(symbol)
            for symbol in symbols
        ]

    # --------------------------------------------------------
    # Tokenize complete sentence
    # --------------------------------------------------------

    def tokenize(self, text):

        tokens = []

        for word in text.split():

            tokens.extend(
                self.tokenize_word(word)
            )

        return tokens

    def encode(self, text):
        tokens = self.tokenize(text)

        ids = []
        for token in tokens:
            if token in self.stoi:
                ids.append(self.stoi[token])
            else:
                # unknown token
                ids.append(self.stoi["<unk>"])

        return ids

    def decode(self, ids):
      byte_values = []

      for idx in ids:
          token = self.itos[idx]

          if token in BPE_SPECIALS:
              continue

          symbol = self.token_to_symbol[token]

          for b in symbol:
              if b == EOW:
                  byte_values.append(ord(" "))
              else:
                  byte_values.append(b)

      # Remove the final space
      if byte_values and byte_values[-1] == ord(" "):
          byte_values.pop()

      return bytes(byte_values).decode("utf-8")

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = data["merges"]
        tokenizer.itos = data["itos"]
        tokenizer.stoi = {
            token: i for i, token in enumerate(tokenizer.itos)
        }
    
        # Initial symbols
        initial_symbols = [(b,) for b in range(256)]
        initial_symbols.append((EOW,))
    
        tokenizer.token_to_symbol = {
            tokenizer.symbol_to_string(symbol): symbol
            for symbol in initial_symbols
        }
    
        # Reconstruct merge rules in training order
        for left_str, right_str in tokenizer.merges:
            left = tokenizer.string_to_symbol(left_str)
            right = tokenizer.string_to_symbol(right_str)
            merged = left + right
    
            tokenizer.merge_rules[(left, right)] = merged
            merged_str = tokenizer.symbol_to_string(merged)
            tokenizer.token_to_symbol[merged_str] = merged
        tokenizer._build_merge_ranks()  
        return tokenizer

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    def save(self, path):

        data = {
            "merges": self.merges,
            "itos": self.itos,
            "vocab_size": self.vocab_size
        }

        with open(
            path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                ensure_ascii=False
            )

        print(
            f"Saved: {path}"
        )

# if __name__ == "__main__":
#     train_text = """
#     Harry went to Hogwarts.
#     Harry met Ron and Hermione.
#     Hogwarts was a magical school.
#     """

#     tokenizer = train_tokenizer(train_text, 5000)

#     print("Vocab size:", tokenizer.vocab_size)

#     text = "Harry went to Hogwarts."

#     ids = tokenizer.encode(text)
#     decoded = tokenizer.decode(ids)

#     print("Text:", text)
#     print("IDs:", ids)
#     print("Decoded:", decoded)
#     print("Match:", text == decoded)


In [7]:
!pip install ijson
import os
import ijson

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

VOCAB_SIZE = 16000
MAX_CHARS_PER_SOURCE = 5_000_000

TOKENIZER_PATH = os.path.join(
    OUTPUT_DIR, "legal_bpe.json"
)


# ------------------------------------------------------------
# Extract text from a record
# ------------------------------------------------------------

def get_text(record):
    if isinstance(record, str):
        return record

    if isinstance(record, dict):
        for key in [
            "text", "content", "judgment",
            "document", "full_text", "body"
        ]:
            if isinstance(record.get(key), str):
                return record[key]

    raise ValueError(
        f"Unable to extract text. Record type: {type(record)}"
    )


# ------------------------------------------------------------
# Stream text from JSON files
# ------------------------------------------------------------

def iter_texts(file_path):
    with open(file_path, "rb") as f:
        for record in ijson.items(f, "item"):
            text = get_text(record)
            if text.strip():
                yield text


# ------------------------------------------------------------
# Create bounded training sample
# ------------------------------------------------------------

def all_documents():
    for source, path in TRAIN_FILES.items():
        print(f"Processing {source}...")

        total_chars = 0

        for text in iter_texts(path):
            remaining = MAX_CHARS_PER_SOURCE - total_chars

            if remaining <= 0:
                break

            text = text[:remaining]

            if text.strip():
                yield text
                total_chars += len(text)

        print(f"{source}: {total_chars:,} characters")


# ------------------------------------------------------------
# Train and save tokenizer
# ------------------------------------------------------------

if TRAIN_NEW_TOKENIZER:
    tokenizer = ByteLevelBPE(vocab_size=VOCAB_SIZE)
    tokenizer.train(all_documents())

    tokenizer.save(TOKENIZER_PATH)

    print(f"Tokenizer saved at: {TOKENIZER_PATH}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 1.7 MB/s eta 0:00:00a 0:00:01
Training BYTE-LEVEL BPE
Processing supreme_court...
supreme_court: 5,000,000 characters
Processing high_courts...
high_courts: 5,000,000 characters
Processing acts...
acts: 5,000,000 characters
Processing tribunals...
tribunals: 5,000,000 characters
Unique words: 112,431
Initial vocabulary size: 261
Merges:  250 | Vocab:   511 | Best pair frequency: 8,421
Merges:  500 | Vocab:   761 | Best pair frequency: 3,564
Merges:  750 | Vocab:  1011 | Best pair frequency: 2,300
Merges: 1000 | Vocab:  1261 | Best pair frequency: 1,657
Merges: 1250 | Vocab:  1511 | Best pair frequency: 1,243
Merges: 1500 | Vocab:  1761 | Best pair frequency: 988
Merges: 1750 | Vocab:  2011 | Best pair frequency: 801
Merges: 2000 | Vocab:  2261 | Best pair frequency: 686
Merges: 2250 | Vocab:  2511 | Best pair frequency: 575
Merges: 2500 | Vocab:  2761 | Best pair frequency: 502
Merges: 2750 | Vocab:  3011 | Best pair frequency:

KeyboardInterrupt: 

In [6]:
# Load the tokenizer: the one trained above, or the saved tokenizer used for the submissions.
tokenizer = ByteLevelBPE.load(
    TOKENIZER_PATH if TRAIN_NEW_TOKENIZER else TOKENIZER_INPUT_PATH
)

print("Vocabulary size:", len(tokenizer.itos))
print("Number of merges:", len(tokenizer.merges))

# Special-token ids (needed by the corpus tokenization step below).
PAD_ID = tokenizer.stoi["<pad>"]   # 0
UNK_ID = tokenizer.stoi["<unk>"]   # 1
BOS_ID = tokenizer.stoi["<s>"]     # 2
EOS_ID = tokenizer.stoi["</s>"]    # 3


Vocabulary size: 16000
Number of merges: 15739


In [10]:
import json
from tqdm.auto import tqdm
from pathlib import Path

# ------------------------------------------------------------
# Count tokens in one source
# ------------------------------------------------------------


def count_source_tokens(tokenizer, file_path):
    total_tokens = 0
    total_docs = 0
    total_chars = 0

    for text in tqdm(
        iter_texts(file_path),
        desc=Path(file_path).name
    ):
        ids = tokenizer.encode(text)

        total_tokens += len(ids)
        total_docs += 1
        total_chars += len(text)

        del ids

    return {
        "documents": total_docs,
        "characters": total_chars,
        "tokens": total_tokens,
    }


# ------------------------------------------------------------
# Count tokens in all sources
# ------------------------------------------------------------

def count_all_sources(tokenizer):
    results = {}

    for source, file_path in TRAIN_FILES.items():
        print(f"\nCounting {source}...")
        results[source] = count_source_tokens(
            tokenizer, file_path
        )

    return results


# ------------------------------------------------------------
# Print and save statistics
# ------------------------------------------------------------

def save_and_print_counts(results, output_path="token_counts.json"):
    total_tokens = sum(
        stats["tokens"] for stats in results.values()
    )

    print("\n" + "=" * 75)
    print(
        f"{'Source':<20}{'Documents':>15}"
        f"{'Characters':>18}{'Tokens':>18}"
    )
    print("=" * 75)

    for source, stats in results.items():
        print(
            f"{source:<20}"
            f"{stats['documents']:>15,}"
            f"{stats['characters']:>18,}"
            f"{stats['tokens']:>18,}"
        )

    print("=" * 75)
    print(f"{'TOTAL':<20}{'':>15}{'':>18}{total_tokens:>18,}")

    with open(output_path, "w") as f:
        json.dump(
            {
                "sources": results,
                "total_tokens": total_tokens,
            },
            f,
            indent=2,
        )

    print(f"\nStatistics saved to {output_path}")


# ------------------------------------------------------------
# Run
# ------------------------------------------------------------
tokenizer._build_merge_ranks()
if COUNT_CORPUS_TOKENS:
    results = count_all_sources(tokenizer)
    save_and_print_counts(results)
else:
    print("Skipped full-corpus token counting (COUNT_CORPUS_TOKENS=False).")


Counting supreme_court...


train_supreme_court.json: 0it [00:00, ?it/s]

KeyboardInterrupt: 

In [11]:
def run_tokenizer_checks(tokenizer):
    print("Vocabulary size:", len(tokenizer.itos))
    assert len(tokenizer.itos) == len(tokenizer.stoi)
    assert len(tokenizer.itos) <= tokenizer.vocab_size

    # Symbol serialization round trips
    symbols = [
        (97,),
        (97, 98),
        (97, 98, EOW),
        (EOW,),
        (240, 159, 152, 128, EOW),
    ]

    for symbol in symbols:
        encoded = tokenizer.symbol_to_string(symbol)
        decoded = tokenizer.string_to_symbol(encoded)
        assert decoded == symbol, (symbol, encoded, decoded)

    # Text round trips: your current tokenizer normalizes whitespace.
    tests = [
        "The Supreme Court held that the appeal is allowed.",
        "Section 302 of the Indian Penal Code.",
        "Article 14 and Article 21.",
        "A.B.C. v. State of X",
        "न्यायालय",
        "AnUnseenWord123",
    ]

    for text in tests:
        ids = tokenizer.encode(text)
        decoded = tokenizer.decode(ids)

        # Compare against the whitespace-normalized text.
        expected = " ".join(text.split())
        assert decoded == expected, (text, decoded, expected)

    # Save/load consistency
    tokenizer.save("tokenizer_check.json")
    loaded = ByteLevelBPE.load("tokenizer_check.json")

    for text in tests:
        assert tokenizer.encode(text) == loaded.encode(text)

    print("All checks passed.")


run_tokenizer_checks(tokenizer)

Vocabulary size: 16000
Saved: tokenizer_check.json
All checks passed.


## Mixing Strategy

In [9]:

import random
from pathlib import Path
from tqdm.auto import tqdm


# ============================================================
# CORPUS MIXING CONFIGURATION
# Change only this dictionary to tune your data strategy.
# ============================================================

MIXING_STRATEGY = {
    "supreme_court": 1.00,
    "high_courts":    0.30,
    "acts":           1.00,
    "tribunals":      0.50,
}

RANDOM_SEED = 42


def mixed_legal_corpus(
    train_files,
    mixing_strategy=MIXING_STRATEGY,
    seed=RANDOM_SEED,
    show_progress=True,
):
    """
    Stream a mixed legal corpus using document-level sampling.

    Parameters
    ----------
    train_files : dict
        Maps source names to their input file paths.
    mixing_strategy : dict
        Sampling fraction for each source, from 0.0 to 1.0.
    seed : int
        Random seed for reproducibility.
    show_progress : bool
        Whether to show progress bars while reading sources.

    Yields
    ------
    str
        Selected documents in a randomized interleaving
        of the source datasets.

    Notes
    -----
    - Sampling is performed independently per document.
    - The selected token proportions are approximate.
    - Documents are not all loaded into memory.
    """

    rng = random.Random(seed)

    # Validate source names and sampling fractions.
    for source, fraction in mixing_strategy.items():
        if source not in train_files:
            raise ValueError(f"Unknown source: {source}")
        if not 0.0 <= fraction <= 1.0:
            raise ValueError(
                f"Invalid sampling fraction for {source}: {fraction}"
            )

    # Create a document stream for each source.
    source_streams = {}

    for source, fraction in mixing_strategy.items():
        if fraction == 0:
            continue

        file_path = train_files[source]

        def selected_documents(
            file_path=file_path,
            fraction=fraction,
            source=source,
        ):
            documents = iter_texts(file_path)

            if show_progress:
                documents = tqdm(
                    documents,
                    desc=f"Sampling {source}",
                )

            for text in documents:
                if not isinstance(text, str) or not text.strip():
                    continue

                # Keep or skip each document independently.
                if rng.random() < fraction:
                    yield text

        source_streams[source] = iter(selected_documents())

    # Randomly interleave the active source streams.
    active_sources = list(source_streams.keys())

    while active_sources:
        source = rng.choice(active_sources)

        try:
            yield next(source_streams[source])
        except StopIteration:
            active_sources.remove(source)

In [10]:
import json

def save_mixed_corpus(
    output_path="mixed_legal_corpus.jsonl",
    train_files=TRAIN_FILES,
    mixing_strategy=MIXING_STRATEGY,
    seed=42,
):
    total_docs = 0
    total_chars = 0

    with open(output_path, "w", encoding="utf-8") as f:
        for text in mixed_legal_corpus(
            train_files=train_files,
            mixing_strategy=mixing_strategy,
            seed=seed,
        ):
            f.write(json.dumps(
                {"text": text},
                ensure_ascii=False
            ) + "\n")

            total_docs += 1
            total_chars += len(text)

    print(f"Documents saved: {total_docs:,}")
    print(f"Characters saved: {total_chars:,}")
    print(f"Output file: {output_path}")

    return {
        "documents": total_docs,
        "characters": total_chars,
        "output_path": output_path,
    }


WORKING_TOKEN_CACHE = os.path.join(OUTPUT_DIR, "tokenized_data.npz")
if os.path.exists(TOKENIZED_CACHE_INPUT) or os.path.exists(WORKING_TOKEN_CACHE):
    print("Tokenized cache found; mixed corpus not rebuilt.")
else:
    mixing_stats = save_mixed_corpus()

Sampling supreme_court: 0it [00:00, ?it/s]

Sampling acts: 0it [00:00, ?it/s]

Sampling high_courts: 0it [00:00, ?it/s]

Sampling tribunals: 0it [00:00, ?it/s]

Documents saved: 1,271,094
Characters saved: 7,954,993,612
Output file: mixed_legal_corpus.jsonl


In [13]:
# #load it


# import json

# CORPUS_PATH = "/kaggle/input/datasets/richierich6/corpus/combined_corpus.jsonl"

# # Inspect a few documents
# with open(CORPUS_PATH, "r", encoding="utf-8") as f:
#     for i, line in enumerate(f):
#         doc = json.loads(line)
#         text = doc["text"]

#         # Tokenize using your custom ByteLevelBPE tokenizer
#         token_ids = tokenizer.encode(text)

#         print(f"Document {i+1}")
#         print("Characters:", len(text))
#         print("Tokens:", len(token_ids))
#         print("Sample text:", text[:200])
#         print("First 20 token IDs:", token_ids[:20])
#         print("-" * 60)

#         if i == 2:
#             break
# print("Vocabulary size:", tokenizer.vocab_size)
# print("Special tokens:", tokenizer.specials)

# print("First 10 stoi entries:", list(tokenizer.stoi.items())[:10])

# print("Last 10 itos entries:")
# for idx, token in enumerate(
#     tokenizer.itos[-10:],
#     start=len(tokenizer.itos) - 10
# ):
#     print(idx, repr(token))

# # Check whether special token IDs exist
# print("BOS token ID:", getattr(tokenizer, "bos_token_id", None))
# print("EOS token ID:", getattr(tokenizer, "eos_token_id", None))
# print("PAD token ID:", getattr(tokenizer, "pad_token_id", None))

Document 1
Characters: 24257
Tokens: 5290
Sample text: PETITIONER: HIS HOLINESS SRI VISHWOTHAMA THIRTHA SWAMIAR OFSODE MUTT Vs.

RESPONDENT: THE STATE OF MYSORE

DATE OF JUDGMENT: 18/12/1961

ACT: Temple-Large number of pilgrims visiting and worshiping-If
First 20 token IDs: [5968, 7545, 1239, 76, 10711, 2423, 73, 13522, 87, 86, 706, 7869, 76, 91, 8072, 13672, 638, 3844, 77, 86]
------------------------------------------------------------
Document 2
Characters: 7784
Tokens: 1412
Sample text: Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area 
First 20 token IDs: [3574, 3843, 8250, 4510, 5379, 1860, 44, 6107, 290, 9937, 4882, 590, 6019, 9897, 520, 515, 496, 273, 765, 2025]
------------------------------------------------------------
Document 3
Characters: 1056
Tokens: 239
Sample text: Court No. - 5 Case :- WRIT - A No. - 7644 of 1992 Petiti

## 6. Create Train and Validation Tokens Set

In [7]:
import json
import random
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# Corpus produced by save_mixed_corpus() above.
train_file_path = "mixed_legal_corpus.jsonl"

# Read the attached cache if present; otherwise build it once in OUTPUT_DIR (writable).
CACHE_PATH = (
    TOKENIZED_CACHE_INPUT
    if Path(TOKENIZED_CACHE_INPUT).exists()
    else os.path.join(OUTPUT_DIR, "tokenized_data.npz")
)

if Path(CACHE_PATH).exists():
    # Reuse previously tokenized data
    cached = np.load(CACHE_PATH)
    train_tokens = cached["train_tokens"].tolist()
    val_tokens = cached["val_tokens"].tolist()

    print("Loaded tokenized data from cache.")

else:
    train_tokens = []
    val_tokens = []
    rng = random.Random(SEED)

    with open(train_file_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Tokenizing corpus"):
            if not line.strip():
                continue

            text = json.loads(line)["text"]
            tokens = tokenizer.encode(text)
            tokens.append(EOS_ID)

            # Split at document level
            if rng.random() < 0.05 and len(val_tokens) < VAL_TOKENS:
                remaining = VAL_TOKENS - len(val_tokens)
                val_tokens.extend(tokens[:remaining])

            elif len(train_tokens) < TRAIN_TOKENS:
                remaining = TRAIN_TOKENS - len(train_tokens)
                train_tokens.extend(tokens[:remaining])

            if (
                len(train_tokens) >= TRAIN_TOKENS
                and len(val_tokens) >= VAL_TOKENS
            ):
                break

    # Save tokens for future reuse
    np.savez(
        CACHE_PATH,
        train_tokens=np.array(train_tokens, dtype=np.int32),
        val_tokens=np.array(val_tokens, dtype=np.int32)
    )

    print("Tokenized data saved to cache.")

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")

Loaded tokenized data from cache.
Train tokens: 200,000,000
Validation tokens: 5,000,000


## 7. Create Token Blocks

In [8]:
def create_blocks(tokens, block_size):
    total_length = (len(tokens) // block_size) * block_size
    tokens = tokens[:total_length]
    return [tokens[i:i + block_size] for i in range(0, total_length, block_size)]

train_blocks = create_blocks(train_tokens, TRAIN_BLOCK_SIZE)
val_blocks = create_blocks(val_tokens, TRAIN_BLOCK_SIZE)

print("Training sequences:", len(train_blocks))
print("Validation sequences:", len(val_blocks))

Training sequences: 97656
Validation sequences: 2441


## 8. Create Datasets

In [9]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"input_ids": train_blocks})
val_dataset = Dataset.from_dict({"input_ids": val_blocks})

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['input_ids'],
    num_rows: 97656
})
Dataset({
    features: ['input_ids'],
    num_rows: 2441
})


In [30]:
# import os

# input_file = "/kaggle/working/mixed_legal_corpus.jsonl"
# output_dir = "/kaggle/working/corpus_parts"

# os.makedirs(output_dir, exist_ok=True)

# max_size = 500 * 1024 * 1024  # 500 MB

# part = 0
# current_size = 0
# out = None

# with open(input_file, "rb") as f:
#     for line in f:
#         if out is None or current_size + len(line) > max_size:
#             if out is not None:
#                 out.close()

#             part_path = os.path.join(
#                 output_dir, f"corpus_{part:02d}.jsonl"
#             )
#             out = open(part_path, "wb")
#             print("Created:", part_path)

#             part += 1
#             current_size = 0

#         out.write(line)
#         current_size += len(line)

# if out is not None:
#     out.close()

Created: /kaggle/working/corpus_parts/corpus_00.jsonl
Created: /kaggle/working/corpus_parts/corpus_01.jsonl
Created: /kaggle/working/corpus_parts/corpus_02.jsonl
Created: /kaggle/working/corpus_parts/corpus_03.jsonl
Created: /kaggle/working/corpus_parts/corpus_04.jsonl
Created: /kaggle/working/corpus_parts/corpus_05.jsonl
Created: /kaggle/working/corpus_parts/corpus_06.jsonl
Created: /kaggle/working/corpus_parts/corpus_07.jsonl
Created: /kaggle/working/corpus_parts/corpus_08.jsonl
Created: /kaggle/working/corpus_parts/corpus_09.jsonl
Created: /kaggle/working/corpus_parts/corpus_10.jsonl
Created: /kaggle/working/corpus_parts/corpus_11.jsonl
Created: /kaggle/working/corpus_parts/corpus_12.jsonl
Created: /kaggle/working/corpus_parts/corpus_13.jsonl
Created: /kaggle/working/corpus_parts/corpus_14.jsonl
Created: /kaggle/working/corpus_parts/corpus_15.jsonl


In [ ]:
print("Train examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

print("Train block length:", len(train_dataset[0]["input_ids"]))
print("Validation block length:", len(val_dataset[0]["input_ids"]))

print("First 20 tokens:", train_dataset[0]["input_ids"][:20])
print("EOS token ID:", EOS_ID)

print(
    "EOS count in first block:",
    train_dataset[0]["input_ids"].count(EOS_ID)
)

## 9. Define The Model

## Architecture 1 - Decoder-only Causal Transformer


In [10]:
PAD_ID = tokenizer.stoi["<pad>"]   # 0
UNK_ID = tokenizer.stoi["<unk>"]   # 1
BOS_ID = tokenizer.stoi["<s>"]     # 2
EOS_ID = tokenizer.stoi["</s>"]    # 3

In [11]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class DecoderConfig:
    vocab_size: int
    context_length: int
    n_embd: int
    n_layer: int
    n_head: int
    embd_pdrop: float = 0.1
    resid_pdrop: float = 0.1
    attn_pdrop: float = 0.1
    layer_norm_epsilon: float = 1e-5
    initializer_range: float = 0.02
    bos_token_id: int = 2
    eos_token_id: int = 3
    pad_token_id: int = 0
    n_inner: int | None = None
    activation_function: str = "gelu_new"
    scale_attn_weights: bool = True
    scale_attn_by_inverse_layer_idx: bool = False
    tie_word_embeddings: bool = True

    @property
    def feed_forward_size(self):
        return self.n_inner if self.n_inner is not None else 4 * self.n_embd


class CausalSelfAttention(nn.Module):
    """Causal multi-head self-attention with a fused QKV projection."""

    def __init__(self, config: DecoderConfig, layer_idx: int):
        super().__init__()
        assert config.n_embd % config.n_head == 0, (
            "n_embd must be divisible by n_head"
        )
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.layer_idx = layer_idx
        self.scale_attn_weights = config.scale_attn_weights
        self.scale_attn_by_inverse_layer_idx = (
            config.scale_attn_by_inverse_layer_idx
        )

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=True)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd, bias=True)
        self.attn_dropout = nn.Dropout(config.attn_pdrop)
        self.resid_dropout = nn.Dropout(config.resid_pdrop)

    def forward(self, hidden_states, attention_mask=None):
            batch_size, seq_len, hidden_size = hidden_states.shape
        
            qkv = self.qkv(hidden_states)
            query, key, value = qkv.split(hidden_size, dim=-1)
        
            def split_heads(tensor):
                return tensor.view(
                    batch_size,
                    seq_len,
                    self.n_head,
                    self.head_dim
                ).transpose(1, 2)
        
            query = split_heads(query)
            key = split_heads(key)
            value = split_heads(value)
        
            scale = None
        
            if self.scale_attn_weights:
                scale = 1.0 / math.sqrt(self.head_dim)
        
            if self.scale_attn_by_inverse_layer_idx:
                if scale is None:
                    scale = 1.0
                scale = scale / float(self.layer_idx + 1)
        
            attn_mask = None
            is_causal = True
        
            if attention_mask is not None:
                key_is_valid = attention_mask.to(
                    device=hidden_states.device,
                    dtype=torch.bool
                )
        
                # [B, 1, 1, L]
                attn_mask = key_is_valid[:, None, None, :]
        
                # Since an explicit mask is supplied, combine causal mask manually
                causal = torch.ones(
                    seq_len,
                    seq_len,
                    dtype=torch.bool,
                    device=hidden_states.device
                ).tril()
        
                attn_mask = attn_mask & causal[None, None, :, :]
                is_causal = False
        
            attended = F.scaled_dot_product_attention(
                query,
                key,
                value,
                attn_mask=attn_mask,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=is_causal,
                scale=scale,
            )
        
            attended = attended.transpose(1, 2).contiguous().view(
                batch_size,
                seq_len,
                hidden_size
            )
        
            return self.resid_dropout(
                self.out_proj(attended)
            )


class DecoderMLP(nn.Module):
    """Position-wise feed-forward block (expand -> activation -> project)."""

    def __init__(self, config: DecoderConfig):
        super().__init__()
        self.fc_in = nn.Linear(
            config.n_embd, config.feed_forward_size, bias=True
        )
        self.fc_out = nn.Linear(
            config.feed_forward_size, config.n_embd, bias=True
        )
        self.dropout = nn.Dropout(config.resid_pdrop)
        self.activation_name = config.activation_function

    def _activation(self, x):
        if self.activation_name == "gelu_new":
            # Tanh approximation of GELU.
            return 0.5 * x * (
                1.0
                + torch.tanh(
                    math.sqrt(2.0 / math.pi)
                    * (x + 0.044715 * torch.pow(x, 3.0))
                )
            )
        if self.activation_name == "gelu":
            return F.gelu(x)
        if self.activation_name == "relu":
            return F.relu(x)
        if self.activation_name in ("silu", "swish"):
            return F.silu(x)
        raise ValueError(
            f"Unsupported activation: {self.activation_name}"
        )

    def forward(self, hidden_states):
        return self.dropout(self.fc_out(self._activation(self.fc_in(hidden_states))))


class DecoderBlock(nn.Module):
    """Pre-norm Transformer block with residual attention and MLP paths."""

    def __init__(self, config: DecoderConfig, layer_idx: int):
        super().__init__()
        self.ln_1 = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.attn = CausalSelfAttention(config, layer_idx)
        self.ln_2 = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.mlp = DecoderMLP(config)

    def forward(self, hidden_states, attention_mask=None):
        hidden_states = hidden_states + self.attn(
            self.ln_1(hidden_states), attention_mask=attention_mask
        )
        hidden_states = hidden_states + self.mlp(self.ln_2(hidden_states))
        return hidden_states


class DecoderOnlyTransformer(nn.Module):
    """Decoder-only causal language model implemented in raw PyTorch."""

    def __init__(self, config: DecoderConfig):
        super().__init__()
        if config.n_embd % config.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        if config.context_length <= 0:
            raise ValueError("context_length must be positive")
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(
            config.context_length, config.n_embd
        )
        self.embedding_dropout = nn.Dropout(config.embd_pdrop)
        self.blocks = nn.ModuleList(
            DecoderBlock(config, layer_idx)
            for layer_idx in range(config.n_layer)
        )
        self.final_ln = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.apply(self._initialize_weights)
        if config.tie_word_embeddings:
            # Weight tying: the output projection shares the token-embedding matrix.
            self.lm_head.weight = self.token_embedding.weight

    def _initialize_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def _validate_inputs(self, input_ids, attention_mask=None):
        if input_ids.ndim != 2:
            raise ValueError("input_ids must have shape [batch, sequence]")
        batch_size, seq_len = input_ids.shape
        if seq_len == 0:
            raise ValueError("input sequence must not be empty")
        assert seq_len <= self.config.context_length, (
            f"sequence length {seq_len} exceeds context length "
            f"{self.config.context_length}"
        )
        if attention_mask is not None:
            if attention_mask.shape != (batch_size, seq_len):
                raise ValueError(
                    "attention_mask must have shape [batch, sequence]"
                )
        return batch_size, seq_len

    def _embed(self, input_ids):
        _, seq_len = input_ids.shape
        position_ids = torch.arange(
            seq_len, dtype=torch.long, device=input_ids.device
        ).unsqueeze(0)
        return self.embedding_dropout(
            self.token_embedding(input_ids)
            + self.position_embedding(position_ids)
        )

    def encode(self, input_ids, attention_mask=None):
        """Return final normalized hidden states with shape [B, L, D]."""
        self._validate_inputs(input_ids, attention_mask)
        hidden_states = self._embed(input_ids)
        for block in self.blocks:
            hidden_states = block(hidden_states, attention_mask)
        return self.final_ln(hidden_states)

    def debug_hidden_states(self, input_ids, attention_mask=None):
        """Expose intermediate states for optional migration debugging."""
        self._validate_inputs(input_ids, attention_mask)
        hidden_states = self._embed(input_ids)
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            hidden_states = block(hidden_states, attention_mask)
            states[f"block_{layer_idx}"] = hidden_states
        states["final_norm"] = self.final_ln(hidden_states)
        return states

    def forward(self, input_ids, attention_mask=None, labels=None):
        hidden_states = self.encode(input_ids, attention_mask)
        logits = self.lm_head(hidden_states)
        loss = None

        if labels is not None:
            if labels.shape != input_ids.shape:
                raise ValueError("labels must have the same shape as input_ids")
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            valid_labels = shift_labels.ne(-100)
            if attention_mask is not None:
                valid_labels = valid_labels & attention_mask[:, 1:].bool()
            
            shift_labels = shift_labels.masked_fill(~valid_labels, -100)
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return {
            "loss": loss,
            "logits": logits,
            "hidden_states": hidden_states,
        }


config = DecoderConfig(
    vocab_size=tokenizer.vocab_size,
    context_length=MODEL_CONTEXT_LENGTH,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    embd_pdrop=0.1,
    resid_pdrop=0.1,
    attn_pdrop=0.1,
    layer_norm_epsilon=1e-5,
    initializer_range=0.02,
    bos_token_id=BOS_ID,
    eos_token_id=EOS_ID,
    pad_token_id=PAD_ID,
    n_inner=None,  # default feed-forward width: 4 * n_embd.
    activation_function="gelu_new",
    scale_attn_weights=True,
    scale_attn_by_inverse_layer_idx=False,
    tie_word_embeddings=True,
)
model = DecoderOnlyTransformer(config)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(f"Head dimension: {config.n_embd // config.n_head}")
print(f"Context length: {config.context_length}")


Model parameters: 50,940,160
Model size: 50.94M
Head dimension: 80
Context length: 2048


### Checkpoint save / load utilities


In [ ]:
# Checkpoint utilities: weights are stored as a plain state_dict plus the model config.
import json
from dataclasses import asdict
from pathlib import Path

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def save_scratch_checkpoint(scratch_model, output_dir, overwrite=False):
    output_dir = Path(output_dir)
    if output_dir.exists() and any(output_dir.iterdir()) and not overwrite:
        raise FileExistsError(
            f"Refusing to overwrite nonempty scratch checkpoint: {output_dir}"
        )
    output_dir.mkdir(parents=True, exist_ok=True)
    state = {
        key: value.detach().cpu().contiguous()
        for key, value in scratch_model.state_dict().items()
    }
    torch.save(state, output_dir / "pytorch_model.bin")
    (output_dir / "config.json").write_text(
        json.dumps(asdict(scratch_model.config), indent=2),
        encoding="utf-8",
    )
    print("Saved scratch checkpoint:", output_dir.resolve())
    return output_dir


def load_scratch_checkpoint(scratch_model, checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    config_path = checkpoint_dir / "config.json"
    if config_path.is_file():
        saved_config = json.loads(config_path.read_text(encoding="utf-8"))
        expected_config = asdict(scratch_model.config)
        mismatched = {
            key: (saved_config.get(key), value)
            for key, value in expected_config.items()
            if saved_config.get(key) != value
        }
        if mismatched:
            raise ValueError(
                f"Scratch checkpoint config mismatch: {mismatched}"
            )

    safe_path = checkpoint_dir / "model.safetensors"
    bin_path = checkpoint_dir / "pytorch_model.bin"
    if safe_path.is_file():
        from safetensors.torch import load_file
        state = load_file(str(safe_path), device="cpu")
    elif bin_path.is_file():
        try:
            state = torch.load(
                bin_path, map_location="cpu", weights_only=True
            )
        except TypeError:
            state = torch.load(bin_path, map_location="cpu")
    else:
        raise FileNotFoundError(
            f"No scratch model weights found in {checkpoint_dir}"
        )
    result = scratch_model.load_state_dict(state, strict=True)
    print(
        "Scratch checkpoint load:",
        f"missing={result.missing_keys}, unexpected={result.unexpected_keys}",
    )
    return scratch_model


In [13]:
# Scratch-model sanity checks: architecture, masking, checkpoint loading,
# forward dimensions, finite loss/logits, and backward gradients.
import tempfile

assert config.vocab_size == tokenizer.vocab_size == len(tokenizer.itos)
assert config.context_length == MODEL_CONTEXT_LENGTH == 2048
assert config.n_embd == N_EMBD == 640
assert config.n_layer == N_LAYER == 8
assert config.n_head == N_HEAD == 8
assert config.n_embd % config.n_head == 0
assert config.activation_function == "gelu_new"
assert config.layer_norm_epsilon == 1e-5
assert config.tie_word_embeddings is True
assert model.lm_head.weight is model.token_embedding.weight

parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
ff_size = config.feed_forward_size
expected_parameter_count = (
    config.vocab_size * config.n_embd
    + config.context_length * config.n_embd
    + config.n_layer
    * (
        3 * config.n_embd * config.n_embd
        + 3 * config.n_embd
        + config.n_embd * config.n_embd
        + config.n_embd
        + 4 * config.n_embd
        + config.n_embd * ff_size
        + ff_size
        + ff_size * config.n_embd
        + config.n_embd
    )
    + 2 * config.n_embd
)
print("Parameter count:", f"{parameter_count:,}")
print("Trainable parameter count:", f"{trainable_parameter_count:,}")
assert parameter_count == expected_parameter_count
assert 50_000_000 <= parameter_count <= 300_000_000

model = model.to(device)
model.eval()
test_ids = torch.tensor(
    [[BOS_ID, 257, 81, 903, 42, EOS_ID], [BOS_ID, 19, 22, 311, 7, EOS_ID]],
    dtype=torch.long,
    device=device,
)
test_mask = torch.ones_like(test_ids)
test_labels = test_ids.clone()
test_labels[0, -1] = PAD_ID

with torch.inference_mode():
    test_outputs = model(
        input_ids=test_ids,
        attention_mask=test_mask,
        labels=test_labels,
    )
assert test_outputs["hidden_states"].shape == (
    2, 6, config.n_embd
)
assert test_outputs["logits"].shape == (
    2, 6, config.vocab_size
)
assert torch.isfinite(test_outputs["logits"]).all()
assert torch.isfinite(test_outputs["loss"])
print("Forward dimensions and finite values: PASS")

# A future-token mutation must not change any earlier-position logits.
with torch.inference_mode():
    original_logits = model(test_ids[:1])["logits"]
    changed_future = test_ids[:1].clone()
    changed_future[0, 4:] = torch.tensor(
        [901, 902], dtype=torch.long, device=device
    )
    changed_logits = model(changed_future)["logits"]
assert torch.allclose(
    original_logits[:, :4], changed_logits[:, :4], atol=1e-6, rtol=1e-6
)
print("Causal future-token masking: PASS")

# A masked padding key must not change a later valid token's representation.
padding_mask = torch.tensor([[0, 1, 1, 1]], dtype=torch.long, device=device)
pad_variant_a = torch.tensor(
    [[PAD_ID, 15, 16, 17]], dtype=torch.long, device=device
)
pad_variant_b = torch.tensor(
    [[1234, 15, 16, 17]], dtype=torch.long, device=device
)
with torch.inference_mode():
    pad_logits_a = model(
        pad_variant_a, attention_mask=padding_mask
    )["logits"][:, -1]
    pad_logits_b = model(
        pad_variant_b, attention_mask=padding_mask
    )["logits"][:, -1]
assert torch.allclose(pad_logits_a, pad_logits_b, atol=1e-6, rtol=1e-6)
print("Attention-mask padding behavior: PASS")

# Loss/backward check on a small batch to keep this cell inexpensive.
model.train()
model.zero_grad(set_to_none=True)
train_labels = test_ids.clone()
train_labels[0, -1] = -100
train_outputs = model(input_ids=test_ids, labels=train_labels)
assert torch.isfinite(train_outputs["loss"])
train_outputs["loss"].backward()
parameters_with_gradients = sum(
    parameter.grad is not None
    for parameter in model.parameters()
    if parameter.requires_grad
)
assert parameters_with_gradients > 0
assert model.blocks[0].attn.qkv.weight.grad is not None
print("Finite loss and backward gradients: PASS")
model.zero_grad(set_to_none=True)
model.eval()

# Save/load a temporary scratch checkpoint and require a strict clean load.
with tempfile.TemporaryDirectory(prefix="decoder_scratch_check_") as temp_dir:
    save_scratch_checkpoint(model, temp_dir)
    reloaded_model = DecoderOnlyTransformer(config).to(device)
    load_scratch_checkpoint(reloaded_model, temp_dir)
    reloaded_model.eval()
    with torch.inference_mode():
        reloaded_logits = reloaded_model(
            input_ids=test_ids, attention_mask=test_mask
        )["logits"]
        original_logits = model(
            input_ids=test_ids, attention_mask=test_mask
        )["logits"]
    assert torch.equal(original_logits, reloaded_logits)
    assert reloaded_model.lm_head.weight is reloaded_model.token_embedding.weight
    del reloaded_model
print("Scratch checkpoint save/load: PASS")
print("Scratch-model sanity checks passed.")


Parameter count: 50,940,160
Trainable parameter count: 50,940,160
Forward dimensions and finite values: PASS
Causal future-token masking: PASS
Attention-mask padding behavior: PASS
Finite loss and backward gradients: PASS
Saved scratch checkpoint: /tmp/decoder_scratch_check_zm86h6dz
Scratch checkpoint load: missing=[], unexpected=[]
Scratch checkpoint save/load: PASS
Scratch-model sanity checks passed.


In [14]:
def data_collator(features):
    input_ids = torch.tensor(
        [feature["input_ids"] for feature in features],
        dtype=torch.long
    )

    return {
        "input_ids": input_ids,
        "labels": input_ids.clone()
    }

## 10. Training Arguments

In [15]:
import torch
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=3,
    per_device_eval_batch_size=3,
    gradient_accumulation_steps=2,

    learning_rate=3e-4,
    weight_decay=0.1,
    warmup_steps=500,
    lr_scheduler_type="cosine",

    fp16=torch.cuda.is_available(),

    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    seed=SEED,
    report_to="none",

    # Helpful for memory management on a T4
    dataloader_num_workers=2,
)

## 11. Training

In [16]:
#benchmark


import time
import torch

model.train()

# Keep a copy of the initial weights: the benchmark below takes real optimizer steps.
_benchmark_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.1
)

BATCH_SIZE = 2
GRAD_ACCUM = 3
NUM_BENCHMARK_STEPS = 10

# Use a few batches from the training data
batches = []
for i in range(NUM_BENCHMARK_STEPS * GRAD_ACCUM):
    start_idx = i * BATCH_SIZE
    batch = torch.tensor(
        train_blocks[start_idx:start_idx + BATCH_SIZE],
        dtype=torch.long,
        device=device
    )
    batches.append(batch)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

optimizer.zero_grad(set_to_none=True)

start_time = time.time()

for step in range(NUM_BENCHMARK_STEPS):
    for micro_step in range(GRAD_ACCUM):
        batch = batches[step * GRAD_ACCUM + micro_step]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=torch.cuda.is_available()
        ):
            outputs = model(input_ids=batch, labels=batch)
            loss = outputs["loss"] / GRAD_ACCUM

        loss.backward()

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.time() - start_time

avg_step_time = elapsed / NUM_BENCHMARK_STEPS
total_optimizer_steps = (
    math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM))
    * EPOCHS
)
estimated_hours = (
    avg_step_time * total_optimizer_steps / 3600
)

print(f"Benchmark time: {elapsed:.2f} seconds")
print(f"Average optimizer step: {avg_step_time:.3f} seconds")
print(f"Estimated training time: {estimated_hours:.2f} hours")

if torch.cuda.is_available():
    peak_memory = torch.cuda.max_memory_allocated()
    print(f"Peak GPU memory: {peak_memory / 1024**3:.2f} GB")

print("Benchmark completed.")

# Restore the pre-benchmark weights so training starts from the same initialization.
model.load_state_dict(_benchmark_state)
del _benchmark_state, optimizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Benchmark time: 6.11 seconds
Average optimizer step: 0.611 seconds
Estimated training time: 8.28 hours
Peak GPU memory: 3.35 GB
Benchmark completed.


In [45]:
print(type(model))
print(model.__class__.__name__)
print(next(model.parameters()).device)

<class '__main__.DecoderOnlyTransformer'>
DecoderOnlyTransformer
cuda:0


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

start_time = time.time()

trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

training_time = time.time() - start_time

tokens_per_epoch = len(train_dataset) * TRAIN_BLOCK_SIZE
total_tokens = tokens_per_epoch * EPOCHS

tokens_per_second = total_tokens / training_time

print(f"Training time: {training_time / 3600:.2f} hours")
print(f"Training speed: {tokens_per_second:,.0f} tokens/sec")

## 12. Perplexity

In [ ]:
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)
print(f"Validation loss: {eval_loss:.6f}")
print(f"Validation perplexity: {perplexity:.4f}")

## 13. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_steps = []
train_loss = []
eval_steps = []
eval_loss = []

for log in logs:
    if ("loss" in log and "step" in log):
        train_steps.append(log["step"])
        train_loss.append(log["loss"])
    if ("eval_loss" in log and "step" in log):
        eval_steps.append(log["step"])
        eval_loss.append(log["eval_loss"])

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_loss, label="Training Loss")
plt.plot(eval_steps, eval_loss, label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Phase 1 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()

print("Loss plot saved to:")
print(LOSS_PLOT_PATH)

## 14. Save Final Model to Kaggle Output

In [ ]:
save_scratch_checkpoint(
    model,
    MODEL_OUTPUT_DIR,
    overwrite=True,
)

print("Final scratch model saved to:")
print(MODEL_OUTPUT_DIR)

# Zip the saved model folder for download.
shutil.make_archive(
    os.path.join(OUTPUT_DIR, "deconly-scratch"),
    "zip",
    MODEL_OUTPUT_DIR
)


## 15. Push Model to Hugging Face Hub

In [ ]:
# model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)

# print("Model and tokenizer pushed successfully.")

## 16. Load Test Data from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

## 18. Load Final Model for Generation

In [ ]:
generation_tokenizer = tokenizer
generation_model = DecoderOnlyTransformer(config)

# Load the final checkpoint saved in section 14.
load_scratch_checkpoint(generation_model, MODEL_OUTPUT_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_model = generation_model.to(device)
generation_model.eval()

print("Checkpoint:", Path(MODEL_OUTPUT_DIR).resolve())
print("Device:", device)
print(
    f"Parameters: "
    f"{sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M"
)


## 19. Generate 128 Words
#### (words, not tokens)

In [ ]:
def _completed_generated_words(token_ids, tokenizer):
    completed = 0
    for token_id in token_ids:
        token = tokenizer.itos[int(token_id)]
        symbol = tokenizer.token_to_symbol.get(token)
        if symbol and symbol[-1] == EOW:
            completed += 1
    return completed


def _banned_ngram_tokens(token_ids, ngram_size):
    if ngram_size <= 0 or len(token_ids) + 1 < ngram_size:
        return set()
    if ngram_size == 1:
        return set(token_ids)
    prefix = tuple(token_ids[-(ngram_size - 1):])
    banned = set()
    for index in range(len(token_ids) - ngram_size + 1):
        ngram = token_ids[index:index + ngram_size]
        if tuple(ngram[:-1]) == prefix:
            banned.add(ngram[-1])
    return banned


def _sample_top_p(logits, temperature=0.7, top_p=0.95):
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero")
    if not 0.0 < top_p <= 1.0:
        raise ValueError("top_p must be in the interval (0, 1]")
    logits = logits.float() / temperature
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probabilities = torch.softmax(sorted_logits, dim=-1)
    cumulative_probabilities = torch.cumsum(sorted_probabilities, dim=-1)
    remove = cumulative_probabilities > top_p
    remove[1:] = remove[:-1].clone()
    remove[0] = False
    logits[sorted_indices[remove]] = -torch.inf
    probabilities = torch.softmax(logits, dim=-1)
    return torch.multinomial(probabilities, num_samples=1).item()


def generate_ids_for_prompt(
    prompt,
    model,
    tokenizer,
    device,
    target_words=128,
    max_new_tokens=MAX_GENERATION_TOKENS,
    temperature=0.7,
    top_p=TOP_P,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
):
    if target_words <= 0:
        raise ValueError("target_words must be positive")
    if max_new_tokens <= 0 or max_new_tokens >= model.config.context_length:
        raise ValueError(
            "max_new_tokens must be positive and smaller than context length"
        )

    prompt_ids = tokenizer.encode(prompt)
    max_prompt_tokens = model.config.context_length - max_new_tokens
    prompt_ids = prompt_ids[-max_prompt_tokens:]
    if not prompt_ids:
        prompt_ids = [BOS_ID]

    generated_ids = []
    model.eval()
    with torch.inference_mode():
        for _ in range(max_new_tokens):
            sequence = (prompt_ids + generated_ids)[
                -model.config.context_length:
            ]
            input_ids = torch.tensor(
                [sequence], dtype=torch.long, device=device
            )
            logits = model(input_ids=input_ids)["logits"][0, -1].clone()

            if repetition_penalty != 1.0:
                if repetition_penalty <= 0:
                    raise ValueError("repetition_penalty must be positive")
                for token_id in set(prompt_ids + generated_ids):
                    score = logits[token_id]
                    logits[token_id] = (
                        score * repetition_penalty
                        if score < 0
                        else score / repetition_penalty
                    )

            banned = _banned_ngram_tokens(
                prompt_ids + generated_ids, no_repeat_ngram_size
            )
            if banned:
                logits[list(banned)] = -torch.inf

            # The fixed 128-word contract takes priority over an early EOS:
            # EOS is available after the quota, while sampling stops as soon
            # as the requested number of complete BPE words has been emitted.
            if _completed_generated_words(generated_ids, tokenizer) < target_words:
                logits[EOS_ID] = -torch.inf
            logits[PAD_ID] = -torch.inf
            logits[BOS_ID] = -torch.inf
            logits[UNK_ID] = -torch.inf

            next_token = _sample_top_p(
                logits, temperature=temperature, top_p=top_p
            )
            if next_token == EOS_ID:
                break
            generated_ids.append(next_token)
            if _completed_generated_words(generated_ids, tokenizer) >= target_words:
                break

    generated_text = tokenizer.decode(generated_ids)
    generated_words = generated_text.split()
    if len(generated_words) != target_words:
        raise RuntimeError(
            f"Generation produced {len(generated_words)} complete words; "
            f"expected exactly {target_words}. Increase max_new_tokens or "
            "inspect the tokenizer word-boundary symbols."
        )
    return generated_ids


def generate_batch(
    prompts,
    model,
    tokenizer,
    device,
    target_words=128,
    temperature=0.7,
    top_p=TOP_P,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
):
    predictions = []
    for prompt in prompts:
        generated_ids = generate_ids_for_prompt(
            prompt=prompt,
            model=model,
            tokenizer=tokenizer,
            device=device,
            target_words=target_words,
            max_new_tokens=MAX_GENERATION_TOKENS,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
        )
        text = tokenizer.decode(generated_ids)
        if len(text.split()) != target_words:
            raise RuntimeError("Generated prediction does not contain 128 words.")
        predictions.append(text)
    return predictions


## 20. Generate Predictions for the Test Set

In [ ]:
# Select one actual test prompt
example_id = list(test_data.keys())[99]
test_prompt = test_data[example_id]["text"]

encoded = tokenizer.encode(test_prompt)[-MODEL_CONTEXT_LENGTH:]
print("Prompt tokens:", len(encoded))
print("Prompt preview:", repr(tokenizer.decode(encoded)[-300:]))

test_output_ids = generate_ids_for_prompt(
    prompt=test_prompt,
    model=generation_model,
    tokenizer=tokenizer,
    device=device,
    target_words=128,
    max_new_tokens=MAX_GENERATION_TOKENS,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
)
generated_text = tokenizer.decode(test_output_ids)
assert len(generated_text.split()) == 128

print("\nGenerated token IDs:", test_output_ids[:30])
print("\nGenerated token count:", len(test_output_ids))
print("\nGenerated text:")
print(repr(generated_text))
print("\nGenerated words:", len(generated_text.split()))


In [ ]:
print("Tokenizer vocab:", len(tokenizer.itos))
print("Model vocab:", generation_model.config.vocab_size)
print("PAD:", PAD_ID)
print("EOS:", EOS_ID)

In [ ]:
predictions = {}
GENERATION_BATCH_SIZE = 32

items = list(test_data.items())
total_examples = len(items)

print(f"Total test examples: {total_examples}")

for start in range(0, total_examples, GENERATION_BATCH_SIZE):
    batch = items[start:start + GENERATION_BATCH_SIZE]

    batch_ids = [str(example_id) for example_id, _ in batch]
    batch_prompts = [record["text"] for _, record in batch]

    batch_predictions = generate_batch(
        prompts=batch_prompts,
        model=generation_model,
        tokenizer=generation_tokenizer,
        device=device,
        target_words=128
    )

    if len(batch_predictions) != len(batch_ids):
        raise RuntimeError(
            f"Expected {len(batch_ids)} predictions, "
            f"received {len(batch_predictions)}"
        )

    for example_id, prediction in zip(batch_ids, batch_predictions):
        predictions[example_id] = {"text": prediction}

    completed = start + len(batch)
    print(
        f"Completed {completed}/{total_examples} "
        f"({100 * completed / total_examples:.1f}%)"
    )

print("Final prediction count:", len(predictions))

## 21. Save prediction.json

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("prediction.json saved:")
print(PREDICTION_PATH)

## 22. Create CodaBench Submission ZIP

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:")
print(SUBMISSION_ZIP)